In [132]:
import pandas as pd
import os
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
import shutil
import numpy as np
import pymorphy2
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GridSearchCV
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold



In [133]:

train = pd.read_csv('C:/Users/Elena/Downloads/train.csv')
test = pd.read_csv('C:/Users/Elena/Downloads/test.csv')

print("Данные загружены!")
print(f"Train: {train.shape}")
print(f"Test: {test.shape}")

Данные загружены!
Train: (41872, 3)
Test: (17946, 2)


In [134]:

def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['clean_review'] = train['review'].apply(clean_text)
test['clean_review'] = test['review'].apply(clean_text)

train['len_review'] = train['clean_review'].str.len()
test['len_review'] = test['clean_review'].str.len()
train['word_count'] = train['clean_review'].str.split().str.len()
test['word_count'] = test['clean_review'].str.split().str.len()
train['orig_len'] = train['review'].str.len()
test['orig_len'] = test['review'].str.len()
train['exclam_count'] = train['review'].str.count(r'!')
test['exclam_count'] = test['review'].str.count(r'!')
train['caps_count'] = train['review'].str.count(r'[A-ZА-Я]')
test['caps_count'] = test['review'].str.count(r'[A-ZА-Я]')
train['question_count'] = train['review'].str.count(r'\?')
test['question_count'] = test['review'].str.count(r'\?')

y = train['sentiment']
print("Признаки созданы!")

Признаки созданы!


In [138]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['clean_review'] = train['review'].apply(clean_text)
test['clean_review'] = test['review'].apply(clean_text)

vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 3), min_df=7, max_df=0.8)
X_train = vectorizer.fit_transform(train['clean_review'])
X_test = vectorizer.transform(test['clean_review'])
y_train = train['sentiment']

model = LogisticRegression(C=2.5, max_iter=1000, solver='liblinear', random_state=42)

scores = cross_val_score(model, X_train, y_train, cv=3, scoring='f1')
print(f"F1 на CV: {scores.mean():.4f} (+/- {scores.std():.4f})")

model.fit(X_train, y_train)
predictions = model.predict(X_test)

submission = pd.DataFrame({'id': test['id'], 'sentiment': predictions})
submission.to_csv('submission.csv', index=False)
print("Сабмит сохранён")

F1 на CV: 0.9317 (+/- 0.0025)
Сабмит сохранён


In [136]:
submission = pd.read_csv('submission.csv')
print(f"Колонки: {list(submission.columns)}")
print(f"Строк в submission: {len(submission)}")
print("Уникальные классы предсказаний:", submission['sentiment'].unique())

Колонки: ['id', 'sentiment']
Строк в submission: 17946
Уникальные классы предсказаний: [1 0]


In [137]:
import os
os.startfile('c:\\Users\\Elena\\Desktop\\BorisovaDaria-1')